# 04 · Balón parado y uso de jugadores

**Qué hace.** Cubre los dos componentes que la página oficial pide y faltaban:
- **Balón parado, a favor y en contra.** Envíos (córners, tiros libres y laterales en zona de peligro), tiros, xG, goles y xG por envío. Además, las zonas de remate: dónde remata el América y dónde le rematan.
- **Uso de jugadores.** Cuántos cambios hace, cuándo hace el primero táctico, cuántos por lesión, de qué línea saca y cuánto rota por torneo.

Compara a Jardine con Solari y Ortiz con el **criterio v2** (`hallazgo.comparar`):
- IC 95 % por bootstrap de partidos sin cruzar 0;
- tamaño ≥ 0.5 DE entre partidos;
- 4 de 5 torneos en la misma dirección;
- FDR de Benjamini-Hochberg dentro de cada familia.

**Qué necesita.** Pasos 02 y 03 corridos (`data/processed/`).

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda menos de un minuto. Las tablas se guardan en `outputs/hallazgos/` (local, no se suben) y la figura en `outputs/exploracion/`.

> IMPOR1: sin el Clausura 2026 (`INCLUIR_PRUEBA = False`).
>
> IMPOR2: solo agregados; el notebook se sube sin salidas (nbstripout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import partidos as fp
from ojo.lentes import balon_parado, hallazgo, jugadores
from ojo.salidas.estilo import color

P = params()
EQUIPO = P["equipo"]["team_id"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
INCLUIR_PRUEBA = False  # IMPOR1
DIR_FIG, DIR_HALLAZGOS = DIR_SALIDAS / "exploracion", DIR_SALIDAS / "hallazgos"
for d in (DIR_FIG, DIR_HALLAZGOS):
    d.mkdir(parents=True, exist_ok=True)

ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
NUEVO, PREVIOS = list(ENTRENADORES.values())[-1], "Predecesores"

partidos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
partidos = partidos[partidos["entrenador_id"].isin(ENTRENADORES)].copy()
partidos["prueba"] = fp.es_prueba(partidos)
if not INCLUIR_PRUEBA:
    partidos = partidos[~partidos["prueba"]]
partidos["entrenador"] = partidos["entrenador_id"].map(ENTRENADORES)
partidos["grupo"] = np.where(partidos["entrenador"] == NUEVO, NUEVO, PREVIOS)
ids = partidos["match_id"].astype(int).tolist()
eventos = almacen.cargar("eventos", ids, columnas=[
    "match_id", "periodo", "indice", "minuto", "posesion", "posesion_equipo_id", "equipo_id", "tipo", "patron", "x", "y",
    "tiro_xg", "tiro_resultado", "sustitucion_resultado", "posicion_id", "jugador_id"])
partidos.groupby("grupo").agg(partidos=("match_id", "size"), torneos=("torneo_corto", lambda s: ", ".join(s.unique())))

## 1. Balón parado

Por partido, a favor y en contra del América. *Envío* = posesión que empieza en córner, tiro libre o lateral en zona de peligro (`framework.md`, "Envío a balón parado").

In [ ]:
env = balon_parado.envios(eventos)
tir = balon_parado.tiros(eventos, env)
bp = (balon_parado.por_partido(env, tir, EQUIPO, partidos["match_id"])
      .merge(partidos[["match_id", "grupo", "torneo_corto", "rival"]], on="match_id"))
METRICAS_BP = ["envios_favor", "xg_favor", "xg_por_envio_favor", "goles_favor",
               "envios_contra", "xg_contra", "xg_por_envio_contra", "goles_contra"]
NOMBRES_BP = {
    "envios_favor": "Envíos a favor", "xg_favor": "xG a favor", "xg_por_envio_favor": "xG por envío a favor",
    "goles_favor": "Goles a favor", "envios_contra": "Envíos en contra", "xg_contra": "xG en contra",
    "xg_por_envio_contra": "xG por envío en contra", "goles_contra": "Goles en contra",
}
bp.groupby("grupo")[METRICAS_BP].mean().T.rename(index=NOMBRES_BP).round(3)

In [ ]:
balon = hallazgo.comparar(bp, METRICAS_BP, NUEVO, PREVIOS, RNG)
balon.to_parquet(DIR_HALLAZGOS / "balon_parado.parquet", index=False)
print(f"Pruebas: {len(balon)} · condiciones 1–3: {balon['pasa_1a3'].sum()} · v2 (con FDR): {balon['pasa'].sum()}")
balon.assign(metrica=balon["metrica"].map(NOMBRES_BP))[
    ["metrica", "nuevo", "previos", "diferencia", "ic_bajo", "ic_alto", "tamano_de", "torneos_misma_direccion", "q", "pasa"]].round(3)

### Zonas de remate y tipo de envío

xG por partido en cada zona (en la orientación de quien tira) y por tipo de envío. *Favor* = remata el América; *contra* = le rematan.

In [ ]:
t = tir.merge(partidos[["match_id", "grupo"]], on="match_id")
t["lado"] = np.where(t["equipo_id"] == EQUIPO, "favor", "contra")
n_partidos = partidos.groupby("grupo").size()


def resumen(por: str) -> pd.DataFrame:
    r = t.groupby(["grupo", "lado", por]).agg(tiros=("xg", "size"), xg=("xg", "sum"), goles=("gol", "sum")).reset_index()
    r["xg_por_partido"] = r["xg"] / r["grupo"].map(n_partidos)
    r["xg_por_tiro"] = r["xg"] / r["tiros"]
    return r


zonas, tipos = resumen("zona"), resumen("tipo")
zonas.to_parquet(DIR_HALLAZGOS / "balon_parado_zonas.parquet", index=False)
tipos.to_parquet(DIR_HALLAZGOS / "balon_parado_tipos.parquet", index=False)
print(tipos.pivot_table(index=["lado", "tipo"], columns="grupo", values="xg_por_partido").round(3))
zonas.pivot_table(index=["lado", "zona"], columns="grupo", values=["xg_por_partido", "xg_por_tiro"]).round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for eje, lado in zip(ejes, ("favor", "contra")):
    tabla = zonas[zonas["lado"] == lado].pivot(index="zona", columns="grupo", values="xg_por_partido").reindex(balon_parado.ZONAS)
    y = np.arange(len(tabla))
    eje.barh(y - .2, tabla[NUEVO].fillna(0), height=.4, color=color("dorado"), label=NUEVO)
    eje.barh(y + .2, tabla[PREVIOS].fillna(0), height=.4, color=color("azul"), label="Solari y Ortiz")
    eje.set_yticks(y, tabla.index)
    eje.set_title(f"xG de balón parado {lado} por partido")
    eje.invert_yaxis()
ejes[0].legend()
fig.savefig(DIR_FIG / "04_balon_parado.png", dpi=130, bbox_inches="tight")
plt.show()

## 2. Uso de jugadores

Cambios por partido. *Tácticos tempranos* = cambios tácticos antes del minuto `jugadores.cambio_temprano_min` de reloj.

In [ ]:
subs = jugadores.sustituciones(eventos, EQUIPO)
uso = jugadores.por_partido(subs, partidos["match_id"]).merge(partidos[["match_id", "grupo", "torneo_corto"]], on="match_id")
METRICAS_USO = ["cambios", "cambios_tacticos", "cambios_lesion", "primer_cambio_tactico", "tacticos_tempranos"]
NOMBRES_USO = {"cambios": "Cambios por partido", "cambios_tacticos": "Cambios tácticos", "cambios_lesion": "Cambios por lesión",
               "primer_cambio_tactico": "Minuto del primer cambio táctico", "tacticos_tempranos": "Cambios tácticos tempranos"}
cambios = hallazgo.comparar(uso, METRICAS_USO, NUEVO, PREVIOS, RNG)
cambios.to_parquet(DIR_HALLAZGOS / "jugadores.parquet", index=False)
print(f"Pruebas: {len(cambios)} · condiciones 1–3: {cambios['pasa_1a3'].sum()} · v2 (con FDR): {cambios['pasa'].sum()}")
cambios.assign(metrica=cambios["metrica"].map(NOMBRES_USO))[
    ["metrica", "nuevo", "previos", "diferencia", "ic_bajo", "ic_alto", "tamano_de", "torneos_misma_direccion", "q", "pasa"]].round(3)

In [ ]:
lineas = (subs[~subs["lesion"]].merge(partidos[["match_id", "grupo"]], on="match_id")
          .groupby("grupo")["linea"].value_counts(normalize=True).rename("proporcion").reset_index())
lineas.to_parquet(DIR_HALLAZGOS / "jugadores_lineas.parquet", index=False)
entrenador_de = partidos.groupby("torneo_corto")["entrenador"].agg(lambda s: s.mode().iloc[0])
orden = partidos.sort_values("fecha")["torneo_corto"].unique()
rot = jugadores.rotacion(eventos, EQUIPO, partidos.set_index("match_id")["torneo_corto"])
rot["entrenador"] = rot["torneo"].map(entrenador_de)
rot = rot.set_index("torneo").reindex(orden).reset_index()
rot.to_parquet(DIR_HALLAZGOS / "jugadores_rotacion.parquet", index=False)
print(lineas.pivot(index="linea", columns="grupo", values="proporcion").round(3))
rot.round(2)

In [ ]:
por_linea = (subs[~subs["lesion"]].dropna(subset=["linea"]).groupby(["match_id", "linea"]).size().unstack(fill_value=0)
             .reindex(partidos["match_id"].unique(), fill_value=0))
por_linea.columns = ["tacticos_" + c.lower() for c in por_linea.columns]
por_linea = por_linea.rename_axis("match_id").reset_index().merge(partidos[["match_id", "grupo", "torneo_corto"]], on="match_id")
LINEAS_T = [c for c in por_linea.columns if c.startswith("tacticos_") and c != "tacticos_portero"]
cambios_linea = hallazgo.comparar(por_linea, LINEAS_T, NUEVO, PREVIOS, RNG)
cambios_linea.to_parquet(DIR_HALLAZGOS / "jugadores_lineas_criterio.parquet", index=False)
print(f"Pruebas: {len(cambios_linea)} · v2 (con FDR): {cambios_linea['pasa'].sum()}")
cambios_linea[["metrica", "nuevo", "previos", "diferencia", "ic_bajo", "ic_alto", "tamano_de", "torneos_misma_direccion", "q", "pasa"]].round(3)

## Lectura

Corrida del 18 oct (AP21–AP25, sin el CL26; 110 partidos de Jardine y 82 de Solari y Ortiz). Los números cambian si cambian los datos o `params.yaml`.

1. **El balón parado no distingue a Jardine.**
    - Ninguna de las 8 pruebas pasa el criterio v2.
    - A favor genera 0.60 xG por partido (0.54 con los predecesores) con menos envíos (28.5 frente a 31.8; el IC no cruza 0, pero el tamaño es de 0.45 DE).
    - En contra concede lo mismo (0.36 frente a 0.37 xG).
2. **Dónde sí cambia: el área chica, en ataque.**
    - Con Jardine, el xG de balón parado que nace en el área chica sube de 0.14 a 0.21 por partido, y cada remate ahí vale más (0.34 frente a 0.24 xG).
    - En contra, el peligro está en el centro del área (0.20 xG por partido) y en el área chica (0.09).
    - Es descriptivo, sin prueba formal. Sirve para la ficha: cada rival se compara con esto.
3. **Uso de jugadores: cuántos cambios y cuándo, igual.** Hace 4.6 cambios por partido (4.4 antes) y el primer cambio táctico llega en el minuto 61 (igual que antes). Ninguna de las 5 pruebas pasa.
4. **Lo que sí lo distingue: cambia la defensa** (criterio v2, 5 de 5 torneos).
    - Hace 1.16 cambios tácticos de defensas por partido, frente a 0.43 de Solari y Ortiz (0.87 DE, q = 0.001), y menos de mediocampistas (1.17 frente a 1.67).
    - Sobre todo laterales y carrileros: 0.75 por partido frente a 0.33. También centrales: 0.40 frente a 0.10.
    - La mediana de esos cambios es el minuto 76.
    - Candidato a principio: *refresca las bandas*. Con los quiebres del notebook 02 (tras una lesión presiona más; las sustituciones tácticas, por sí solas, no cambian el juego en 10 minutos), se lee como gestión de piernas más que como golpe táctico.
5. **Rotación.** Usa entre 27 y 34 jugadores por torneo y 15.3–15.9 por partido, casi lo mismo que sus predecesores (23–30 y 15.1–15.7).